## Group G:
1. [teammate]
2. Akhmad Fauzi Nurulhamzah

# Part 2: Streaming application using Spark Structured Streaming
In this task, you will implement Spark Structured Streaming to consume the data from task 1 and perform a prediction.
Important:
-	This task uses PySpark Structured Streaming with PySpark Dataframe APIs and PySpark ML.
-	You also need your pipeline model from A2A to make predictions and persist the results.

1.	Write code to create a SparkSession, which 1) uses four cores with a proper application name; 2) use the Melbourne timezone; 3) ensure a checkpoint location has been set.

This task uses `local[4]` so that Spark can execute tasks across four local
processing cores, as required by the assignment. This follows the parallel
processing concept discussed in the unit, where a workload can be divided
into smaller tasks and processed concurrently using multiple processing
resources. In a local shared-memory environment, this is consistent with the
Symmetric Multi Processor (SMP) architecture described by Taniar et al. (2008),
where multiple processors share common memory and secondary storage.

The `spark.sql.shuffle.partitions` setting is set to 8 so that operations such
as aggregations and joins can distribute data across multiple partitions and
be processed in parallel across the four available cores. This reflects
intra-operation parallelism, where partitioned data enables operations such
as `groupBy` and joins to be executed concurrently (Taniar et al., 2008).
This is particularly relevant to the later streaming transformations applied
to the incoming weather data.

The session also uses the `Australia/Melbourne` timezone to maintain
consistent timestamp processing and a checkpoint directory to support
progress tracking and recovery in Structured Streaming.

In [2]:
# ==============================
# Kafka connector setup
# ==============================
import os
import time

os.environ["PYSPARK_SUBMIT_ARGS"] = (
    "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.1 pyspark-shell"
)

# ==============================
# Import statements
# ==============================
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType
)
from pyspark.ml import PipelineModel

# ==============================
# Kafka configuration
# ==============================
KAFKA_BOOTSTRAP = "192.168.1.7:9092"   # Kafka broker address
TOPIC_IN        = "weather5s"

# ==============================
# Path configuration
# ==============================
A2A_DIR    = os.environ.get("A2B_A2A_DIR", ".")
DATA       = os.path.join(A2A_DIR, "dataset")
MODEL_PATH = os.path.join(A2A_DIR, "models", "best_model")

CHECKPOINT = "_ckpt_task2"
OUTPUT     = "_out_task2"

# Create directories for streaming checkpoints and outputs
os.makedirs(CHECKPOINT, exist_ok=True)
os.makedirs(OUTPUT, exist_ok=True)

# ==============================
# 1. Create SparkSession
# ==============================
spark = (
    SparkSession.builder
    .master("local[4]")                                           # four cores
    .appName("ITI5202_A2B_Streaming")                              # application name
    .config("spark.sql.session.timeZone", "Australia/Melbourne")   # Melbourne timezone
    .config("spark.sql.streaming.checkpointLocation", CHECKPOINT)  # checkpoint location
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.showConsoleProgress", "false")
    .config(
        "spark.sql.ansi.enabled",
        "false"
    )  # invalid numeric casts become null, useful for missing weather values
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

# ==============================
# Sanity checks
# ==============================
print("Spark version :", spark.version)
print("Timezone      :", spark.conf.get("spark.sql.session.timeZone"))
print(
    "Checkpoint    :",
    spark.conf.get("spark.sql.streaming.checkpointLocation")
)
print("Model path    :", MODEL_PATH, "| exists:", os.path.exists(MODEL_PATH))
print("Data path     :", DATA, "| exists:", os.path.exists(DATA))

Spark version : 4.1.1
Timezone      : Australia/Melbourne
Checkpoint    : _ckpt_task2
Model path    : ./models/best_model | exists: True
Data path     : ./dataset | exists: True


2.	Write code to define the data schema for the data files, following the data types suggested in the metadata file. Load the static datasets (e.g. building information) into data frames. (You can reuse your code from 2A.)

The schema for `new_building_information.csv` is defined explicitly using
data types consistent with the metadata and the A2A model. The static building
data provide the building-level information required for the prediction
pipeline, including `primary_use`, `square_feet`, `floor_count`, `year_built`,
`latent_y`, `latent_s`, and `latent_r`.

The identifiers `site_id` and `building_id` are retained to associate each
building with its corresponding site and streaming weather data. The `row_id`
column is removed because it is an identifier from the source dataset and is
not used as an input feature by the A2A model.

The building information is loaded as a static DataFrame and cached because
it is reused when joining the incoming weather stream with the corresponding
building information.

In [3]:
# ==============================
# 2. Data schemas and static datasets
# ==============================

# Schema for new building information, using types compatible with the A2A model.
building_schema = StructType([
    StructField("site_id", IntegerType()),
    StructField("building_id", IntegerType()),
    StructField("primary_use", StringType()),
    StructField("square_feet", IntegerType()),
    StructField("floor_count", IntegerType()),
    StructField("row_id", IntegerType()),
    StructField("year_built", IntegerType()),
    StructField("latent_y", DoubleType()),
    StructField("latent_s", DoubleType()),
    StructField("latent_r", DoubleType()),
])

# Load static building information; cached because it is reused in stream joins
buildings = (
    spark.read
    .option("header", True)
    .schema(building_schema)
    .csv(f"{DATA}/new_building_information.csv")
    .drop("row_id")
    .cache()
)

print(
    "Static buildings loaded:", buildings.count(),
    "| sites:", buildings.select("site_id").distinct().count()
)

buildings.printSchema()
buildings.show(3)

Static buildings loaded: 449 | sites: 16
root
 |-- site_id: integer (nullable = true)
 |-- building_id: integer (nullable = true)
 |-- primary_use: string (nullable = true)
 |-- square_feet: integer (nullable = true)
 |-- floor_count: integer (nullable = true)
 |-- year_built: integer (nullable = true)
 |-- latent_y: double (nullable = true)
 |-- latent_s: double (nullable = true)
 |-- latent_r: double (nullable = true)

+-------+-----------+-----------+-----------+-----------+----------+--------+---------+--------+
|site_id|building_id|primary_use|square_feet|floor_count|year_built|latent_y| latent_s|latent_r|
+-------+-----------+-----------+-----------+-----------+----------+--------+---------+--------+
|     10|       1017| Technology|     109263|          6|      1971|    29.0|  4.26031|     4.0|
|      4|        587| Technology|      53234|          5|      1949|    51.0|4.0271864|     3.0|
|     13|       1081| Industrial|      11470|          1|      2005|     5.0|4.0595636|   

3.	Using the Kafka topic from the producer in Task 1, ingest the streaming data into Spark Streaming, assuming all data comes in the String format. Except for the 'weather_ts' column, you shall receive it as an Int type. Load the new building information CSV file into a dataframe. Then, the data frames should be transformed into the proper formats following the metadata file schema, similar to assignment 2A.

Weather data from Task 1 are ingested from the Kafka topic `weather5s` using
Spark Structured Streaming. Incoming fields are initially read as strings,
except `weather_ts` as an integer, and are then converted to the data types
required by the metadata and A2A model.

The Kafka source uses `startingOffsets="earliest"` so that existing records
in the topic can also be processed when a new streaming query starts. This is
useful when the producer has already started publishing data before the Spark
query begins. According to the Apache Spark documentation, `startingOffsets`
controls the starting position for a newly created Kafka streaming query
(Apache Spark, 2026).

In [4]:
# ==============================
# 3. Ingest streaming data from Kafka
# ==============================

# Incoming Kafka data: all fields are String except weather_ts, which is Integer.
json_schema = StructType([
    StructField("site_id", StringType()),
    StructField("timestamp", StringType()),
    StructField("air_temperature", StringType()),
    StructField("cloud_coverage", StringType()),
    StructField("dew_temperature", StringType()),
    StructField("sea_level_pressure", StringType()),
    StructField("wind_direction", StringType()),
    StructField("wind_speed", StringType()),
    StructField("weather_ts", IntegerType()),
])

# Read weather stream from the Kafka topic created in Task 1
raw = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
    .option("subscribe", TOPIC_IN)
    .option("startingOffsets", "earliest")  # process existing Kafka records when the query first starts
    .option("maxOffsetsPerTrigger", "500")
    .load()
)

# Parse Kafka message value from JSON
parsed = (
    raw
    .select(
        F.from_json(
            F.col("value").cast("string"),
            json_schema
        ).alias("data")
    )
    .select("data.*")
)

# Transform fields into the proper data types used in A2A
typed = (
    parsed
    .withColumn("site_id", F.col("site_id").cast("int"))
    .withColumn(
        "timestamp",
        F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss.SSS")
    )
    .withColumn("air_temperature", F.col("air_temperature").cast("double"))
    .withColumn("cloud_coverage", F.col("cloud_coverage").cast("double"))
    .withColumn("dew_temperature", F.col("dew_temperature").cast("double"))
    .withColumn("sea_level_pressure", F.col("sea_level_pressure").cast("double"))
    .withColumn("wind_direction", F.col("wind_direction").cast("double"))
    .withColumn("wind_speed", F.col("wind_speed").cast("double"))
)

print("Streaming source ready. isStreaming =", typed.isStreaming)
typed.printSchema()

Streaming source ready. isStreaming = True
root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- cloud_coverage: double (nullable = true)
 |-- dew_temperature: double (nullable = true)
 |-- sea_level_pressure: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- weather_ts: integer (nullable = true)



4.	Use a watermark on weather_ts, if data points are received 5 seconds late, discard the data.

The watermark allows Spark to limit
the amount of state maintained for late-arriving streaming data and enables
data arriving beyond the allowed event-time delay to be discarded during
stateful operations.

In [5]:
# ==============================
# 4. Watermark on weather_ts
# ==============================

event_df = (
    typed
    .withColumn(
        "event_time",
        F.timestamp_seconds("weather_ts")
    )
    .withWatermark("event_time", "5 seconds")
)

print("Watermark set on event_time: 5 seconds.")
event_df.printSchema()

Watermark set on event_time: 5 seconds.
root
 |-- site_id: integer (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- air_temperature: double (nullable = true)
 |-- cloud_coverage: double (nullable = true)
 |-- dew_temperature: double (nullable = true)
 |-- sea_level_pressure: double (nullable = true)
 |-- wind_direction: double (nullable = true)
 |-- wind_speed: double (nullable = true)
 |-- weather_ts: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)



5.	Perform the necessary transformation you used in A2A. (note: every student may have used different features, feel free to reuse the code you have written in A2A. If you built an end-to-end pipeline, you can ignore this task.)

### Rebuilding the A2A Model Features

The transformations used in A2A are reproduced so that the streaming data
contain the same 15 input features used during model training. These include
three categorical features (`primary_use`, `season`, and `interval_period`)
and twelve numerical features (`square_feet`, `floor_count`, `year_built`,
`latent_y`, `latent_s`, `latent_r`, `air_temperature`, `dew_temperature`,
`sea_level_pressure`, `wind_speed`, `cloud_coverage`, and `is_off_day`).

Historical weather data are used to recreate the A2A imputation values and
site-month peak/off-peak season mapping. The streaming weather data are then
imputed and aggregated into the same 6-hour intervals used in A2A before being
joined with the static building information. Finally, `interval_period` and
`is_off_day` are recreated so that the resulting data are compatible with the
saved A2A pipeline.

In [6]:
# ==============================
# 5. Rebuild the transformations used in A2A
# ==============================

# 5.1 Load historical weather for imputation and season mapping
weather_hist_schema = StructType([
    StructField("site_id", IntegerType()),
    StructField("timestamp", StringType()),
    StructField("air_temperature", DoubleType()),
    StructField("cloud_coverage", DoubleType()),
    StructField("dew_temperature", DoubleType()),
    StructField("sea_level_pressure", DoubleType()),
    StructField("wind_direction", DoubleType()),
    StructField("wind_speed", DoubleType()),
])

w_hist = (
    spark.read
    .option("header", True)
    .schema(weather_hist_schema)
    .csv(f"{DATA}/weather.csv")
)


# 5.2 Recreate A2A imputation values
MEAN_COLS = [
    "air_temperature",
    "dew_temperature",
    "sea_level_pressure"
]

MEDIAN_COLS = [
    "wind_speed",
    "cloud_coverage"
]

impute_means = {
    c: float(v)
    for c, v in (
        w_hist
        .select([F.avg(c).alias(c) for c in MEAN_COLS])
        .first()
        .asDict()
        .items()
    )
}

impute_medians = {
    c: float(w_hist.approxQuantile(c, [0.5], 0.01)[0])
    for c in MEDIAN_COLS
}

impute_values = {
    **impute_means,
    **impute_medians
}


# 5.3 Recreate A2A peak/off-peak season mapping
hist_with_month = (
    w_hist
    .withColumn(
        "hist_timestamp",
        F.to_timestamp("timestamp", "yyyy-MM-dd HH:mm:ss.SSS")
    )
    .withColumn("month", F.month("hist_timestamp"))
)

month_avg = (
    hist_with_month
    .groupBy("site_id", "month")
    .agg(F.avg("air_temperature").alias("avg_temp"))
)

hot_window = (
    Window
    .partitionBy("site_id")
    .orderBy(F.desc("avg_temp"))
)

cold_window = (
    Window
    .partitionBy("site_id")
    .orderBy(F.asc("avg_temp"))
)

season_map = (
    month_avg
    .withColumn("hot_rank", F.rank().over(hot_window))
    .withColumn("cold_rank", F.rank().over(cold_window))
    .withColumn(
        "season",
        F.when(
            (F.col("hot_rank") <= 3) |
            (F.col("cold_rank") <= 3),
            "peak"
        ).otherwise("off-peak")
    )
    .select("site_id", "month", "season")
    .cache()
)


# 5.4 Prepare the streaming weather data
weather_prepared = (
    event_df
    .na.fill(impute_values)
    .withColumn("month", F.month("timestamp"))
    .withColumn("date", F.to_date("timestamp"))
    .join(
        season_map,
        ["site_id", "month"],
        "left"
    )
    .na.fill({"season": "off-peak"})
)


# 5.5 Create the same 6-hour intervals used in A2A
weather_prepared = (
    weather_prepared
    .withColumn(
        "interval_start_hour",
        (F.hour("timestamp") / 6).cast("int") * 6
    )
    .withColumn(
        "interval_start",
        F.to_timestamp(
            F.concat_ws(
                " ",
                F.col("date").cast("string"),
                F.lpad(
                    F.col("interval_start_hour").cast("string"),
                    2,
                    "0"
                )
            ),
            "yyyy-MM-dd HH"
        )
    )
)


# 5.6 Aggregate hourly weather to 6-hour granularity
weather_6h = (
    weather_prepared
    .groupBy(
        "site_id",
        "event_time",
        "interval_start",
        "season"
    )
    .agg(
        F.avg("air_temperature").alias("air_temperature"),
        F.avg("dew_temperature").alias("dew_temperature"),
        F.avg("sea_level_pressure").alias("sea_level_pressure"),
        F.avg("wind_speed").alias("wind_speed"),
        F.avg("cloud_coverage").alias("cloud_coverage")
    )
)


# 5.7 Join building information and recreate model features
features = (
    weather_6h
    .join(
        buildings,
        "site_id",
        "inner"
    )
    .withColumn(
        "interval_period",
        F.when(F.hour("interval_start") < 6, "00-05")
        .when(F.hour("interval_start") < 12, "06-11")
        .when(F.hour("interval_start") < 18, "12-17")
        .otherwise("18-23")
    )
    .withColumn(
        "is_off_day",
        F.when(
            F.dayofweek("interval_start").isin([1, 7]),
            1
        ).otherwise(0)
    )
    .withColumn(
        "date",
        F.to_date("interval_start")
    )
)


# 5.8 Sanity check: all model inputs are available
MODEL_INPUTS = [
    "primary_use",
    "season",
    "interval_period",
    "square_feet",
    "floor_count",
    "year_built",
    "latent_y",
    "latent_s",
    "latent_r",
    "air_temperature",
    "dew_temperature",
    "sea_level_pressure",
    "wind_speed",
    "cloud_coverage",
    "is_off_day"
]

missing = [
    c for c in MODEL_INPUTS
    if c not in features.columns
]

print(
    "Model inputs present:",
    len(MODEL_INPUTS) - len(missing),
    "/",
    len(MODEL_INPUTS)
)

print(
    "Missing:",
    missing if missing else "none"
)

Model inputs present: 15 / 15
Missing: none


6.	Load your pipeline model and perform the following aggregations:
a)	Print the prediction from your model as a stream comes in.
b)	Every 7 seconds, print the total energy consumption for each 6-hour interval, aggregated by building, and print 20 records. (Note: This is simulating energy data each day in a week)
c)	Every 14 seconds, for each site, print the daily total energy consumption.

The saved A2A pipeline predicts `log1p(energy_consumption)`, so `expm1` is
applied to convert the prediction back to the original energy-consumption
scale. Negative values are clipped to zero.

For Task 6a, incoming model predictions are printed for each micro-batch.
For Task 6b, one prediction is retained for each building and 6-hour interval,
because each prediction already represents the total predicted energy for that
building during that interval.

For Task 6c, the latest building-level 6-hour predictions are accumulated and
summed by `site_id` and `date` to produce the daily predicted energy total for
each site.

In [7]:
# ==============================
# 6. Apply the model and generate streaming predictions
# ==============================

import time
from collections import defaultdict

from pyspark.ml import PipelineModel
from pyspark.sql import functions as F


# ==============================
# 6.1 Load the saved A2A pipeline
# ==============================

model = PipelineModel.load(MODEL_PATH)

print(
    "Loaded pipeline:",
    [type(stage).__name__ for stage in model.stages]
)


# ==============================
# 6.2 Generate predictions
# ==============================

# The A2A model was trained using log1p(energy_consumption),
# so convert predictions back to the original energy scale.

pred = (
    model
    .transform(features)
    .withColumn(
        "predicted_energy",
        F.greatest(
            F.expm1(F.col("prediction")),
            F.lit(0.0)
        )
    )
    .select(
        "building_id",
        "site_id",
        "date",
        "interval_start",
        "interval_period",
        "event_time",
        "predicted_energy"
    )
)

print("Prediction stream ready:", pred.isStreaming)


# ==============================
# Variables used across batches
# ==============================

# Keep one prediction for each building and 6-hour interval
latest_predictions = {}

last_6b_time = [0]
last_6c_time = [0]


# ==============================
# Process each micro-batch
# ==============================

def process_predictions(batch_df, batch_id):
    """Print 6a for this batch, 6b every 7s, and 6c every 14s."""

    if batch_df.isEmpty():
        return

    batch_df.persist()

    try:

        now = time.time()


        # ==============================
        # 6a. Print predictions as stream comes in
        # ==============================

        print("\n" + "=" * 65)
        print(f"6a | Incoming predictions | Batch {batch_id}")
        print("=" * 65)

        (
            batch_df
            .select(
                "building_id",
                "site_id",
                "date",
                "interval_start",
                "interval_period",
                "predicted_energy"
            )
            .show(
                20,
                truncate=False
            )
        )


        # ==============================
        # Prepare one prediction for each
        # building and 6-hour interval
        # ==============================

        building_6h = (
            batch_df
            .select(
                "building_id",
                "site_id",
                "date",
                "interval_start",
                "interval_period",
                "predicted_energy"
            )
            .dropDuplicates([
                "building_id",
                "interval_start"
            ])
        )


        # ==============================
        # 6b. Every 7 seconds
        # ==============================

        if now - last_6b_time[0] >= 7:

            print("\n" + "=" * 65)
            print("6b | Building energy per 6-hour interval")
            print("=" * 65)

            (
                building_6h
                .orderBy(
                    "date",
                    "interval_start",
                    "building_id"
                )
                .show(
                    20,
                    truncate=False
                )
            )

            last_6b_time[0] = now


        # ==============================
        # Store latest building predictions
        # ==============================

        rows = (
            building_6h
            .select(
                "building_id",
                "site_id",
                "date",
                "interval_start",
                "predicted_energy"
            )
            .collect()
        )

        for row in rows:

            key = (
                row["building_id"],
                row["site_id"],
                str(row["date"]),
                str(row["interval_start"])
            )

            # Overwrite rather than add.
            # This avoids double-counting the same interval.
            latest_predictions[key] = float(
                row["predicted_energy"]
            )


        # ==============================
        # 6c. Every 14 seconds
        # ==============================

        if now - last_6c_time[0] >= 14:

            site_daily = defaultdict(float)

            for key, energy in latest_predictions.items():

                (
                    building_id,
                    site_id,
                    date,
                    interval_start
                ) = key

                site_daily[
                    (site_id, date)
                ] += energy


            result_rows = [
                (
                    site_id,
                    date,
                    total_energy
                )
                for (site_id, date), total_energy
                in site_daily.items()
            ]


            if result_rows:

                result_df = spark.createDataFrame(
                    result_rows,
                    [
                        "site_id",
                        "date",
                        "total_energy"
                    ]
                )

                print("\n" + "=" * 65)
                print("6c | Daily total energy by site")
                print("=" * 65)

                (
                    result_df
                    .orderBy(
                        "date",
                        "site_id"
                    )
                    .show(
                        20,
                        truncate=False
                    )
                )

            last_6c_time[0] = now


    finally:

        batch_df.unpersist()


# ==============================
# Start streaming query
# ==============================

q_all = (
    pred
    .writeStream
    .outputMode("append")
    .foreachBatch(process_predictions)
    .trigger(processingTime="7 seconds")
    .option(
        "checkpointLocation",
        f"{CHECKPOINT}/q6_all"
    )
    .queryName("q6_all")
    .start()
)

print("\nQ6 streaming started.")
print("Active:", q_all.isActive)
print("Exception:", q_all.exception())

Loaded pipeline: ['StringIndexerModel', 'OneHotEncoderModel', 'VectorAssembler', 'GBTRegressionModel']
Prediction stream ready: True

Q6 streaming started.
Active: True
Exception: None

6a | Incoming predictions | Batch 4
+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|708        |5      |2022-01-01|2022-01-01 12:00:00|12-17          |29.487912381150036|
|705        |5      |2022-01-01|2022-01-01 12:00:00|12-17          |35.64851673784263 |
|704        |5      |2022-01-01|2022-01-01 12:00:00|12-17          |103.77009996796663|
|702        |5      |2022-01-01|2022-01-01 12:00:00|12-17          |27.62198237044994 |
|701        |5      |2022-01-01|2022-01-01 12:00:00|12-17          |853.2778170019772 |
|699        |5      |2022-01-01|2022-01-01 12:00:00|12-17          |228.20

+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|4          |0      |2022-01-02|2022-01-02 00:00:00|00-05          |2.974666142014831 |
|7          |0      |2022-01-02|2022-01-02 00:00:00|00-05          |9.532458086823057 |
|11         |0      |2022-01-02|2022-01-02 00:00:00|00-05          |3.620665784701545 |
|14         |0      |2022-01-02|2022-01-02 00:00:00|00-05          |52.40498883307434 |
|15         |0      |2022-01-02|2022-01-02 00:00:00|00-05          |3.1073618340351525|
|19         |0      |2022-01-02|2022-01-02 00:00:00|00-05          |0.0               |
|25         |0      |2022-01-02|2022-01-02 00:00:00|00-05          |3.0927940553088122|
|35         |0      |2022-01-02|2022-01-02 00:00:00|00-05          |0.7061771226767068|
|42         |0      |2022-01-02|


6a | Incoming predictions | Batch 8
+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|106        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |0.9723594482150777|
|155        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |115.17519297412508|
|152        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |17.150866109553135|
|150        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |136.45958077063347|
|149        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |140.18534781498576|
|148        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |1921.9061181966104|
|147        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |178.24681232609407|
|125        |1      |2022-01-06|2022-01-06 00:00:00|00-05          |187.44499897645

+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|1022       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |1109.43210710376  |
|1018       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |25391.124209725476|
|1013       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |383.72901824629366|
|1009       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |95.29946568474173 |
|1006       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |288.3044323095795 |
|1004       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |760.7286239713089 |
|1021       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |7486.069206155662 |
|1002       |10     |2022-01-09|2022-01-09 12:00:00|12-17          |2.625154962746035 |
|1027       |10     |2022-01-09|

+-----------+-------+----------+-------------------+---------------+-------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy   |
+-----------+-------+----------+-------------------+---------------+-------------------+
|4          |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.0                |
|7          |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.7508196483511211 |
|11         |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.0                |
|14         |0      |2022-01-11|2022-01-11 00:00:00|00-05          |2.7507806963447727 |
|15         |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.0                |
|19         |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.0                |
|25         |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.0                |
|35         |0      |2022-01-11|2022-01-11 00:00:00|00-05          |0.0                |
|42         |0      |

...

+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|99         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |52.811224241697836|
|94         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |1900.9264353508972|
|91         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |2378.926845981269 |
|84         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |204.5682986306249 |
|83         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |236.13797812659527|
|76         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |9067.705652140994 |
|75         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |1663.1325185849498|
|68         |0      |2022-06-13|2022-06-13 18:00:00|18-23          |80.88188110531675 |
|60         |0      |2022-06-13|

+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|4          |0      |2022-06-15|2022-06-15 00:00:00|00-05          |1557.3163231183878|
|7          |0      |2022-06-15|2022-06-15 00:00:00|00-05          |8939.19810387458  |
|11         |0      |2022-06-15|2022-06-15 00:00:00|00-05          |1484.4430444581958|
|14         |0      |2022-06-15|2022-06-15 00:00:00|00-05          |17421.575853245973|
|15         |0      |2022-06-15|2022-06-15 00:00:00|00-05          |736.4355638143262 |
|19         |0      |2022-06-15|2022-06-15 00:00:00|00-05          |1373.1204555886347|
|25         |0      |2022-06-15|2022-06-15 00:00:00|00-05          |974.7462371492894 |
|35         |0      |2022-06-15|2022-06-15 00:00:00|00-05          |1519.4510496235657|
|42         |0      |2022-06-15|


6a | Incoming predictions | Batch 134
+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|1216       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |60.15566004600957 |
|1210       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |541.786949874997  |
|1209       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |4118.716194164145 |
|1201       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |1113.9119358379328|
|1195       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |288.7366816739019 |
|1194       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |322.14045429198336|
|1192       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |3331.652709125404 |
|1189       |13     |2022-06-19|2022-06-19 12:00:00|12-17          |2087.82784354


6c | Daily total energy by site
+-------+----------+------------------+
|site_id|date      |total_energy      |
+-------+----------+------------------+
|0      |2022-01-01|25549.563871727758|
|1      |2022-01-01|24893.636960583575|
|2      |2022-01-01|96573.74491823654 |
|3      |2022-01-01|202490.97642898464|
|4      |2022-01-01|220216.39500356212|
|5      |2022-01-01|29273.31433819669 |
|6      |2022-01-01|187906.62970168525|
|7      |2022-01-01|385008.9533958627 |
|8      |2022-01-01|1946.5334960594066|
|9      |2022-01-01|578661.6021930404 |
|10     |2022-01-01|189991.10142528635|
|11     |2022-01-01|83407.5006833207  |
|12     |2022-01-01|149267.37052640956|
|13     |2022-01-01|3311524.126820884 |
|14     |2022-01-01|386810.87397943094|
|15     |2022-01-01|270097.10095205245|
|0      |2022-01-02|8871.9390185244   |
|1      |2022-01-02|23652.863145175048|
|2      |2022-01-02|79676.75013801896 |
|3      |2022-01-02|253826.92807364435|
+-------+----------+------------------+
only sh

+-----------+-------+----------+-------------------+---------------+------------------+
|building_id|site_id|date      |interval_start     |interval_period|predicted_energy  |
+-----------+-------+----------+-------------------+---------------+------------------+
|1322       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |5.855483464438486 |
|1320       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |17.082984840827113|
|1316       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |369.95460113352283|
|1280       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |787.2571623451365 |
|1293       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |6676.227642779608 |
|1314       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |1922.9946352559614|
|1302       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |801.0978389858509 |
|1300       |14     |2022-06-24|2022-06-24 00:00:00|00-05          |361.723898642124  |
|1299       |14     |2022-06-24|

7.	Save the data from 6 to Parquet files as streams. (Hint: Parquet files support streaming writing/reading. The file keeps updating while new batches arrive.)

The outputs generated in Task 6 are continuously saved to separate Parquet
locations using a Structured Streaming query with `foreachBatch`. For each
micro-batch, the incoming predictions (6a) and one prediction for each building
and 6-hour interval (6b) are appended to Parquet.

The latest building-level predictions are also maintained to calculate the
daily predicted energy total for each site (6c), which is written every
14 seconds. A checkpoint is used to track streaming progress and support
recovery when the query is restarted.

In [8]:
# ==============================
# 7. Save Task 6 outputs to Parquet
# ==============================

PARQUET_6A = f"{OUTPUT}/predictions_6a"
PARQUET_6B = f"{OUTPUT}/building_6h_6b"
PARQUET_6C = f"{OUTPUT}/site_daily_6c"

CHECKPOINT_Q7 = f"{CHECKPOINT}/parquet_task7"

q7_latest_predictions = {}
q7_last_6c_time = [0]


def save_task7_outputs(batch_df, batch_id):
    """Append this batch's 6a/6b rows to Parquet, and save 6c every 14s."""

    if batch_df.isEmpty():
        return

    batch_df.persist()

    try:
        now = time.time()

        # 7.1 Save 6a predictions
        predictions_6a = batch_df.select(
            "building_id",
            "site_id",
            "date",
            "interval_start",
            "interval_period",
            "event_time",
            "predicted_energy"
        )

        predictions_6a.write.mode("append").parquet(PARQUET_6A)


        # 7.2 Save 6b building 6-hour predictions
        building_6h = (
            predictions_6a
            .dropDuplicates(["building_id", "interval_start"])
        )

        building_6h.write.mode("append").parquet(PARQUET_6B)


        # Store latest values for 6c
        rows = (
            building_6h
            .select(
                "site_id",
                "date",
                "building_id",
                "interval_period",
                "predicted_energy"
            )
            .collect()
        )

        for row in rows:
            key = (
                row["site_id"],
                row["date"],
                row["building_id"],
                row["interval_period"]
            )

            q7_latest_predictions[key] = float(
                row["predicted_energy"]
            )


        # 7.3 Save 6c every 14 seconds
        if now - q7_last_6c_time[0] >= 14:

            site_daily = defaultdict(float)

            for (
                site_id,
                date,
                _building_id,
                _interval_period
            ), energy in q7_latest_predictions.items():

                site_daily[(site_id, date)] += energy


            result_rows = [
                (site_id, date, total_energy)
                for (site_id, date), total_energy
                in site_daily.items()
            ]


            if result_rows:

                result_df = spark.createDataFrame(
                    result_rows,
                    ["site_id", "date", "total_energy"]
                )

                result_df = (
                    result_df
                    .withColumn("batch_id", F.lit(batch_id))
                    .withColumn(
                        "updated_at",
                        F.current_timestamp()
                    )
                )

                result_df.write.mode("append").parquet(
                    PARQUET_6C
                )

            q7_last_6c_time[0] = now

    finally:
        batch_df.unpersist()


q_parquet = (
    pred
    .writeStream
    .outputMode("append")
    .foreachBatch(save_task7_outputs)
    .option("checkpointLocation", CHECKPOINT_Q7)
    .trigger(processingTime="7 seconds")
    .queryName("q7_parquet")
    .start()
)

print("\nTask 7 stream started:", q_parquet.isActive)
print("Saving 6a to:", PARQUET_6A)
print("Saving 6b to:", PARQUET_6B)
print("Saving 6c to:", PARQUET_6C)


Task 7 stream started: True
Saving 6a to: _out_task2/predictions_6a
Saving 6b to: _out_task2/building_6h_6b
Saving 6c to: _out_task2/site_daily_6c


In [9]:
# ==============================
# Let q_all and q_parquet run, then stop both together
# ==============================

# Check every 5 seconds so the streams can be interrupted and stopped cleanly.

RUN_SECONDS = 1200

print(f"Running for up to {RUN_SECONDS}s (Kernel > Interrupt any time once you've seen enough) ...")

try:
    for _ in range(RUN_SECONDS // 5):
        time.sleep(5)

except KeyboardInterrupt:
    print("\nInterrupted early.")


# Stop q_all and q_parquet together -- see the markdown note above for why.
for q in (q_all, q_parquet):

    try:
        q.stop()

    except Exception as ex:
        print("stop failed:", type(ex).__name__)

print("Queries stopped.")

Running for up to 1200s (Kernel > Interrupt any time once you've seen enough) ...

Interrupted early.
Queries stopped.


8.	Read the parquet files from task 7 as data streams and send them to Kafka topics with appropriate names.
(Note: You shall read the parquet files as a streaming data frame and send messages to the Kafka topic when new data appears in the parquet file.)

The Parquet outputs from Task 7 are read as streaming DataFrames using
`readStream`, so newly added records can be processed as they appear. Each
row is converted to JSON and published to a separate Kafka topic for the
incoming predictions (6a), building-level 6-hour predictions (6b), and daily
site totals (6c).

Separate checkpoints are used for each query to track streaming progress and
support recovery.

In [10]:
# ==============================
# 8. Read Parquet streams and send to Kafka
# ==============================

TOPIC_6A = "prediction_stream"
TOPIC_6B = "building_6h_energy"
TOPIC_6C = "site_daily_energy"

CHECKPOINT_KAFKA_6A = f"{CHECKPOINT}/kafka_6a"
CHECKPOINT_KAFKA_6B = f"{CHECKPOINT}/kafka_6b"
CHECKPOINT_KAFKA_6C = f"{CHECKPOINT}/kafka_6c"


# 8.1 Get schemas from Task 7 outputs
schema_6a = spark.read.parquet(PARQUET_6A).schema
schema_6b = spark.read.parquet(PARQUET_6B).schema
schema_6c = spark.read.parquet(PARQUET_6C).schema


# 8.2 Read Parquet files as streaming DataFrames
stream_6a = (
    spark.readStream
    .schema(schema_6a)
    .format("parquet")
    .load(PARQUET_6A)
)

stream_6b = (
    spark.readStream
    .schema(schema_6b)
    .format("parquet")
    .load(PARQUET_6B)
)

stream_6c = (
    spark.readStream
    .schema(schema_6c)
    .format("parquet")
    .load(PARQUET_6C)
)


# 8.3 Convert rows to JSON
kafka_6a = stream_6a.select(
    F.to_json(F.struct("*")).alias("value")
)

kafka_6b = stream_6b.select(
    F.to_json(F.struct("*")).alias("value")
)

kafka_6c = stream_6c.select(
    F.to_json(F.struct("*")).alias("value")
)


# 8.4 Send streams to Kafka
q_kafka_6a = (
    kafka_6a
    .writeStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        KAFKA_BOOTSTRAP
    )
    .option("topic", TOPIC_6A)
    .option(
        "checkpointLocation",
        CHECKPOINT_KAFKA_6A
    )
    .outputMode("append")
    .queryName("q8_kafka_6a")
    .start()
)


q_kafka_6b = (
    kafka_6b
    .writeStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        KAFKA_BOOTSTRAP
    )
    .option("topic", TOPIC_6B)
    .option(
        "checkpointLocation",
        CHECKPOINT_KAFKA_6B
    )
    .outputMode("append")
    .queryName("q8_kafka_6b")
    .start()
)


q_kafka_6c = (
    kafka_6c
    .writeStream
    .format("kafka")
    .option(
        "kafka.bootstrap.servers",
        KAFKA_BOOTSTRAP
    )
    .option("topic", TOPIC_6C)
    .option(
        "checkpointLocation",
        CHECKPOINT_KAFKA_6C
    )
    .outputMode("append")
    .queryName("q8_kafka_6c")
    .start()
)


print(
    "8.1 | 6a ->",
    TOPIC_6A,
    "| active:",
    q_kafka_6a.isActive
)

print(
    "8.2 | 6b ->",
    TOPIC_6B,
    "| active:",
    q_kafka_6b.isActive
)

print(
    "8.3 | 6c ->",
    TOPIC_6C,
    "| active:",
    q_kafka_6c.isActive
)

8.1 | 6a -> prediction_stream | active: True
8.2 | 6b -> building_6h_energy | active: True
8.3 | 6c -> site_daily_energy | active: True


#### Verifying the Kafka Outputs

The Kafka output queries are allowed to run before being stopped cleanly.
The three Kafka topics are then read from the earliest available offsets to
verify that the Parquet streams from Task 7 have been successfully published.

In [ ]:
# ==============================
# Stop and verify Kafka output streams
# ==============================

# Allow the Parquet streams time to publish their records to Kafka.

print("Publishing to Kafka topics for 300s ...")

time.sleep(300)

for q in (q_kafka_6a, q_kafka_6b, q_kafka_6c):

    try:
        q.stop()

    except Exception as ex:
        print("stop failed:", type(ex).__name__)

print("Republish queries stopped.")


# Verify that messages were published to each Kafka topic
for topic in (TOPIC_6A, TOPIC_6B, TOPIC_6C):

    check = (
        spark.read
        .format("kafka")
        .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP)
        .option("subscribe", topic)
        .option("startingOffsets", "earliest")
        .load()
    )

    print(f"messages in {topic}: {check.count()}")

spark.stop()

print("Spark session stopped.")

#### References

Apache Spark. (2026). Structured Streaming + Kafka integration guide. Apache Spark. https://spark.apache.org/docs/latest/streaming/structured-streaming-kafka-integration.html

Taniar, D., Leung, C. H. C., Rahayu, W., & Goel, S. (2008). *High-Performance Parallel Database Processing and Grid Databases*. John Wiley & Sons.